In [ ]:
import os
import csv
import time
import pycdlib

In [ ]:
# ---------- Configuration ----------

SOURCE_DIR = r"G:\thecall\test"
DEST_DIR   = r"G:\c_out"
CSV_PATH   = os.path.join(DEST_DIR, "inventory.csv")
ERROR_LOG  = os.path.join(DEST_DIR, "errors.log")

MAX_FILENAME_LEN = 200  # truncate filenames longer than this (before extension)


In [ ]:
# ---------- Utilities ----------

def safe_path(path):
    """Prepend the Windows long-path prefix so paths > 260 chars work."""
    path = os.path.abspath(path)
    if not path.startswith("\\\\?\\"):
        return "\\\\?\\" + path
    return path


def clean_name(name):
    """Remove characters illegal on Windows and strip ISO version suffix (;1).

    Windows silently rejects directory/file names that end with a space or
    period even when using the long-path prefix, so we rstrip both.
    """
    if ";" in name:
        name = name[:name.rfind(";")]
    name = "".join(c if c not in '<>:"/\\|?*' else "_" for c in name)
    return name.rstrip(" .")


def truncate_name(name, max_len=MAX_FILENAME_LEN):
    """Truncate a filename while preserving its extension."""
    root, ext = os.path.splitext(name)
    if len(name) <= max_len:
        return name
    return root[: max_len - len(ext)] + ext


def detect_path_type(iso):
    """
    Choose the best path namespace for this ISO:
      'joliet_path' - disc has Joliet extensions (long, Unicode-friendly names)
      'iso_path'    - plain ISO 9660 (8.3 or slightly relaxed names)
    Rock Ridge is skipped; these discs don't have it.
    """
    if iso.joliet_vd is not None:
        return "joliet_path"
    return "iso_path"

In [ ]:
# ---------- Date extraction ----------
#
# pycdlib exposes two distinct date structures depending on context:
#
# DirectoryRecordDate  (ISO 9660 / plain iso_path records)
#   Integer fields: years_since_1900, month, day, hour, minute, second, gmtoffset
#
# VolumeDescriptorDate (Joliet volume descriptor dates, and sometimes
#   returned for Joliet directory records)
#   17-byte ASCII string "YYYYMMDDHHmmsscc+oo", accessible as .date_str
#
# We try both forms so we never silently miss a date.

def _parse_dir_record_date(dt):
    """Parse a DirectoryRecordDate (integer fields, year offset from 1900)."""
    try:
        year = getattr(dt, "years_since_1900", None)
        if year is None or year == 0:
            return ""
        return "{:04d}-{:02d}-{:02d} {:02d}:{:02d}:{:02d}".format(
            year + 1900, dt.month, dt.day,
            dt.hour, dt.minute, dt.second,
        )
    except Exception:
        return ""


def _parse_vd_date(dt):
    """Parse a VolumeDescriptorDate (17-byte ASCII string via .date_str)."""
    date_str = getattr(dt, "date_str", None)
    if not date_str:
        return ""
    try:
        s = date_str.decode("ascii").strip("\x00 ")
        if len(s) >= 14 and s[:14].isdigit():
            return "{}-{}-{} {}:{}:{}".format(
                s[0:4], s[4:6], s[6:8],
                s[8:10], s[10:12], s[12:14],
            )
    except Exception:
        pass
    return ""


def _date_to_epoch(date_str):
    """Convert a 'YYYY-MM-DD HH:MM:SS' string to a Unix timestamp, or None."""
    if not date_str:
        return None
    try:
        return time.mktime(time.strptime(date_str, "%Y-%m-%d %H:%M:%S"))
    except Exception:
        return None


def get_all_dates(iso, iso_file_path, path_type):
    """
    Return a dict of every date value pycdlib can give us for this file.

    CSV columns produced:
      date_primary   - date from the namespace used for extraction (Joliet or ISO)
      date_secondary - date from the other namespace (cross-check; often identical
                       but sometimes one is populated when the other is blank)
      date_source    - which namespace yielded the best date

    We also stamp the best available date back onto the extracted file via
    os.utime() so Windows Explorer shows the original date modified.
    """
    results = {
        "date_primary":   "",
        "date_secondary": "",
        "date_source":    "",
    }

    # --- Primary namespace ---
    try:
        rec = iso.get_record(**{path_type: iso_file_path})
        dt = rec.date
        # Try VolumeDescriptorDate form first (Joliet), then DirectoryRecordDate
        parsed = _parse_vd_date(dt) or _parse_dir_record_date(dt)
        results["date_primary"] = parsed
    except Exception:
        pass

    # --- Secondary namespace (cross-check) ---
    alt_type = "iso_path" if path_type == "joliet_path" else "joliet_path"
    try:
        rec_alt = iso.get_record(**{alt_type: iso_file_path})
        dt_alt = rec_alt.date
        parsed_alt = _parse_vd_date(dt_alt) or _parse_dir_record_date(dt_alt)
        results["date_secondary"] = parsed_alt
    except Exception:
        pass  # alt namespace may not exist on this disc; that is fine

    # Record which source gave us a date
    if results["date_primary"]:
        results["date_source"] = path_type
    elif results["date_secondary"]:
        results["date_source"] = alt_type

    return results


def stamp_file_date(out_path, dates):
    """
    Write the best available date back to the extracted file's mtime/atime
    so Windows Explorer shows the original date rather than today's date.
    """
    best = dates["date_primary"] or dates["date_secondary"]
    epoch = _date_to_epoch(best)
    if epoch:
        try:
            os.utime(safe_path(out_path), (epoch, epoch))
        except Exception:
            pass  # non-fatal; metadata is still in the CSV


def log_error(message):
    with open(ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(message + "\n")


In [ ]:
# ---------- Core extraction ----------

def extract_iso(iso_path, output_dir, writer):
    iso = pycdlib.PyCdlib()
    iso.open(iso_path)

    iso_name  = os.path.basename(iso_path)
    path_type = detect_path_type(iso)
    print(f"  Path namespace: {path_type}")

    for dir_path, dirs, files in iso.walk(**{path_type: "/"}):
        for file in files:
            iso_file_path = dir_path.rstrip("/") + "/" + file

            try:
                clean_file = truncate_name(clean_name(file))

                rel_dir = dir_path.lstrip("/")
                if rel_dir:
                    rel_dir = os.path.join(*[clean_name(p) for p in rel_dir.split("/") if p])
                else:
                    rel_dir = "."

                out_dir  = os.path.join(output_dir, rel_dir)
                out_path = os.path.join(out_dir, clean_file)

                os.makedirs(safe_path(out_dir), exist_ok=True)

                iso.get_file_from_iso(
                    local_path=safe_path(out_path),
                    **{path_type: iso_file_path},
                )

                dates = get_all_dates(iso, iso_file_path, path_type)
                stamp_file_date(out_path, dates)

                writer.writerow({
                    "iso_file":       iso_name,
                    "file_name":      clean_file,
                    "original_name":  file,
                    "folder":         rel_dir,
                    "full_path":      out_path,
                    "date_primary":   dates["date_primary"],
                    "date_secondary": dates["date_secondary"],
                    "date_source":    dates["date_source"],
                    "path_type":      path_type,
                })

            except Exception as e:
                log_error(f"{iso_name} | {iso_file_path} | {str(e)}")

    iso.close()


In [ ]:
# ---------- Main pipeline ----------

def main():
    os.makedirs(DEST_DIR, exist_ok=True)

    fieldnames = [
        "iso_file", "file_name", "original_name",
        "folder", "full_path",
        "date_primary", "date_secondary", "date_source",
        "path_type",
    ]

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for root, _, files in os.walk(SOURCE_DIR):
            for file in files:
                if file.lower().endswith(".iso"):
                    iso_path   = os.path.join(root, file)
                    out_folder = os.path.join(DEST_DIR, os.path.splitext(file)[0])

                    print(f"Processing: {iso_path}")
                    os.makedirs(out_folder, exist_ok=True)

                    try:
                        extract_iso(iso_path, out_folder, writer)
                    except Exception as e:
                        log_error(f"FATAL | {iso_path} | {str(e)}")
                        print(f"  ERROR (skipped): {e}")

    print("Done.")


In [ ]:

if __name__ == "__main__":
    main()
